# D+7 유지 예측 프로토타입

> **목적**: 가입 초기 24시간 내 사용자 행동을 바탕으로 D+7 유지 여부를 예측하고, 사용자별 상대적 이탈 위험도를 산출한다.

### 분석 설계
- **분석 단위**: 사용자 1명 = 1행
- **타깃**: `retained_d7` (D+7 유지 여부, 0/1)
- **기준 시점**: `first_ping_at`
- **피처 관찰 구간**: 최초 PING 이후 초기 24시간
- **검증 방식**: `first_ping_at` 기준 시간순 Train / Validation / Test 분할
- **전처리 원칙**: 전처리기는 Train 데이터에만 `fit`
- **비교 모델**: Dummy → Logistic Regression → Random Forest → LightGBM
- **주요 평가 지표**: ROC-AUC, PR-AUC, Log Loss, Brier Score
- **최종 산출물**: 유지확률(`retention_probability`), 이탈 위험점수(`risk_score`), 상대적 위험구간(`risk_segment`)

### 노트북 실행 흐름
1. 데이터 로드 및 품질 점검
2. 시간순 Train / Validation / Test 분할
3. 모델 입력 피처 및 전처리 정의
4. Baseline / 후보 모델 비교
5. Ablation Test
6. 시간순 교차검증 기반 튜닝
7. 최종 Test 평가 및 위험점수 생성

> **주의**: Test 데이터는 최종 모델 선택이 끝난 뒤 마지막 1회 평가에만 사용한다.


## 1. 데이터 연결 및 로드


In [ ]:
from google.cloud import bigquery
from google.colab import auth

import numpy as np
import pandas as pd

# Google 계정 인증
auth.authenticate_user()

PROJECT_ID = "your-gcp-project"
RANDOM_STATE = 42
TRAIN_RATIO = 0.70
VALID_RATIO = 0.15

client = bigquery.Client(project=PROJECT_ID)

print("BigQuery 연결 완료")


In [ ]:
training_query = """
SELECT *
FROM `your-gcp-project.mart.mart_ml_d7_training`
ORDER BY first_ping_at, user_id
"""

training_df = client.query(training_query).to_dataframe()

print("행 수:", len(training_df))
print("컬럼 수:", len(training_df.columns))
print("고유 사용자 수:", training_df["user_id"].nunique())

training_df.head()

## 2. 데이터 품질 점검

모델링 전에 사용자 중복, 타깃 이상, 결측, 중복·상수성 컬럼과 피처 논리를 확인한다.


In [ ]:
basic_qa = pd.DataFrame({
    "check": [
        "전체 행 수",
        "고유 사용자 수",
        "중복 사용자 수",
        "타깃 결측 수",
        "타깃 0·1 외 값 수",
    ],
    "value": [
        len(training_df),
        training_df["user_id"].nunique(),
        training_df["user_id"].duplicated().sum(),
        training_df["retained_d7"].isna().sum(),
        (~training_df["retained_d7"].isin([0, 1])).sum(),
    ],
})

basic_qa

### 2-1. 타깃 분포


In [ ]:
target_distribution = (
    training_df["retained_d7"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("retained_d7")
    .reset_index(name="user_count")
)

target_distribution["user_pct"] = (
    target_distribution["user_count"]
    / len(training_df)
    * 100
).round(2)

target_distribution

**현재 결과 해석**
- 전체 4,844명 중 D+7 유지 2,347명(48.45%), 비유지 2,497명(51.55%)으로 심한 클래스 불균형은 아니다.
- 따라서 Accuracy 하나만으로 모델을 평가하지 않고, 순위 성능과 확률 품질을 함께 본다.


### 2-2. 컬럼별 결측
- `time_to_first_receive_hours` 는 첫 PING 수신 자체가 없으면 시간을 계산할 수 없으니까 NULL인 것이 정상
- `time_to_first_hint_hours`도 마찬가지로 힌트를 열지 않은 사용자는 “첫 힌트까지 걸린 시간” 자체가 존재하지 않아서 NULL


In [ ]:
missing_summary = (
    training_df
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

missing_summary["missing_rate_pct"] = (
    missing_summary["missing_count"]
    / len(training_df)
    * 100
).round(2)

missing_summary = (
    missing_summary
    .query("missing_count > 0")
    .sort_values(
        "missing_rate_pct",
        ascending=False
    )
)

missing_summary

### 2-3. 상수·저다양성 컬럼


In [ ]:
unique_summary = pd.DataFrame({
    "dtype": training_df.dtypes.astype(str),
    "unique_count": training_df.nunique(dropna=False),
})

unique_summary["unique_rate_pct"] = (
    unique_summary["unique_count"]
    / len(training_df)
    * 100
).round(2)

unique_summary.sort_values("unique_count").head(20)

### 2-4. 완전히 동일한 컬럼 탐색


In [ ]:
duplicate_column_pairs = []

columns = training_df.columns.tolist()

for i, left_col in enumerate(columns):
    for right_col in columns[i + 1:]:
        if training_df[left_col].equals(
            training_df[right_col]
        ):
            duplicate_column_pairs.append(
                (left_col, right_col)
            )

duplicate_column_pairs

### 2-5. 카테고리 피처 논리 검증


In [ ]:
category_cols = [
    "category_0_ping_count_24h",
    "category_1_ping_count_24h",
    "category_2_ping_count_24h",
    "category_3_ping_count_24h",
    "category_4_ping_count_24h",
]

feature_logic_qa = pd.DataFrame({
    "check": [
        "전체 PING 수 불일치",
        "카테고리 매핑 수 불일치",
        "카테고리 비율 범위 오류",
        "카테고리 다양성 범위 오류",
        "대표 카테고리 비율 범위 오류",
    ],
    "error_count": [
        (
            training_df["ping_sent_count_24h"]
            != training_df["total_question_ping_count_24h"]
        ).sum(),

        (
            training_df["categorized_ping_count_24h"]
            + training_df["uncategorized_ping_count_24h"]
            != training_df["total_question_ping_count_24h"]
        ).sum(),

        (
            training_df[category_cols].sum(axis=1)
            != training_df["categorized_ping_count_24h"]
        ).sum(),

        (
            ~training_df["category_diversity_24h"]
            .between(0, 1)
        ).sum(),

        (
            training_df["max_category_share_24h"].notna()
            & ~training_df["max_category_share_24h"]
            .between(0, 1)
        ).sum(),
    ],
})

feature_logic_qa

### 2-6. 시간 분포 및 타깃 변화


In [ ]:
time_distribution = (
    training_df
    .assign(
        first_ping_month=(
            pd.to_datetime(training_df["first_ping_at_kst"])
            .dt.to_period("M")
            .astype(str)
        )
    )
    .groupby("first_ping_month", as_index=False)
    .agg(
        user_count=("user_id", "count"),
        retained_count=("retained_d7", "sum"),
        retained_rate=("retained_d7", "mean"),
    )
)

time_distribution["retained_rate"] = (
    time_distribution["retained_rate"] * 100
).round(2)

time_distribution

**체크 포인트**
- 월별 유지율이 일정하지 않다(특히 후반부 표본 수가 매우 작음).
- 이런 시간 변화가 있으므로 랜덤 분할보다 시간순 검증이 프로젝트 목적에 더 적합하다.


In [ ]:
print(
    "최초 first_ping_at:",
    training_df["first_ping_at_kst"].min()
)

print(
    "마지막 first_ping_at:",
    training_df["first_ping_at_kst"].max()
)

## 3. 시간순 Train / Validation / Test 분할

무작위 분할 대신 `first_ping_at` 순서로 70% / 15% / 15%를 나눈다. 미래 사용자 정보가 과거 학습 구간으로 섞이지 않도록 하기 위한 설계다.


In [ ]:
# first_ping_at 기준 시간순 정렬
modeling_df = (
    training_df
    .sort_values(
        ["first_ping_at", "user_id"]
    )
    .reset_index(drop=True)
)

n_rows = len(modeling_df)

train_end_idx = int(n_rows * TRAIN_RATIO)
valid_end_idx = int(n_rows * (TRAIN_RATIO + VALID_RATIO))

train_df = modeling_df.iloc[:train_end_idx].copy()
valid_df = modeling_df.iloc[train_end_idx:valid_end_idx].copy()
test_df = modeling_df.iloc[valid_end_idx:].copy()

print("Train:", len(train_df))
print("Validation:", len(valid_df))
print("Test:", len(test_df))


In [ ]:
def summarize_split(df, split_name):
    return {
        "split": split_name,
        "user_count": len(df),
        "start_at": df["first_ping_at_kst"].min(),
        "end_at": df["first_ping_at_kst"].max(),
        "retained_count": df["retained_d7"].sum(),
        "retained_rate_pct": round(
            df["retained_d7"].mean() * 100,
            2
        ),
    }


split_summary = pd.DataFrame([
    summarize_split(train_df, "Train"),
    summarize_split(valid_df, "Validation"),
    summarize_split(test_df, "Test"),
])

split_summary

In [ ]:
assert (
    train_df["first_ping_at"].max()
    <= valid_df["first_ping_at"].min()
)

assert (
    valid_df["first_ping_at"].max()
    <= test_df["first_ping_at"].min()
)

assert (
    len(train_df)
    + len(valid_df)
    + len(test_df)
    == len(modeling_df)
)

assert train_df["first_ping_at"].is_monotonic_increasing
assert valid_df["first_ping_at"].is_monotonic_increasing
assert test_df["first_ping_at"].is_monotonic_increasing

print("시간순 분할 검증 완료")


## 4. 모델 입력 피처 정의


In [ ]:
# 사용자 기본 피처
basic_features = [
    "gender",
    "grade",
    "class_num",
    "school_type",
    "student_count",
]

# 초기 24시간 PING 행동 피처
ping_features = [
    "ping_sent_count_24h",
    "ping_received_count_24h",
    "unique_sent_users_24h",
    "unique_received_users_24h",
    "active_hours_24h",
    "send_receive_ratio_24h",
]

# 초기 수신·힌트 피처
hint_features = [
    "had_received_before_t0",
    "first_receive_in_24h",
    "time_to_first_receive_hours",
    "had_hint_before_t0",
    "first_hint_opened_in_24h",
    "time_to_first_hint_hours",
]

# 질문 카테고리 행동 피처
category_features = [
    "category_count_24h",
    "dominant_category_24h",
    "max_category_share_24h",
    "category_diversity_24h",
    "category_0_ping_count_24h",
    "category_1_ping_count_24h",
    "category_2_ping_count_24h",
    "category_3_ping_count_24h",
    "category_4_ping_count_24h",
]

feature_groups = {
    "basic": basic_features,
    "ping": ping_features,
    "hint": hint_features,
    "category": category_features,
}

feature_columns = (
    basic_features
    + ping_features
    + hint_features
    + category_features
)

print("전체 모델 입력 피처 수:", len(feature_columns))

for group_name, columns in feature_groups.items():
    print(f"{group_name}: {len(columns)}개")

In [ ]:
excluded_feature_reasons = {
    "school_id": "집단 식별자·암기 위험",
    "group_id": "집단 식별자·암기 위험",
    "activation_match_flag": "QA 전용 컬럼",
    "total_question_ping_count_24h": "ping_sent_count_24h와 동일",
    "categorized_ping_count_24h": "카테고리별 PING 합계와 중복",
    "uncategorized_ping_count_24h": "매핑 QA 성격",
    "category_mapping_rate_24h": "매핑 품질 QA 성격",
    "dominant_cluster_id_24h": "dominant_category_24h와 동일 정보",
    "received_before_first_hint": "first_hint_opened_in_24h와 동일",
}

pd.Series(
    excluded_feature_reasons,
    name="제외 사유"
).to_frame()

### 4-1. X / y 분리


In [ ]:
categorical_features = [
    "gender",
    "grade",
    "class_num",
    "school_type",
    "dominant_category_24h",
]

numeric_features = [
    col
    for col in feature_columns
    if col not in categorical_features
]

X_train = train_df[feature_columns].copy()
X_valid = valid_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()

y_train = train_df["retained_d7"].astype(int).copy()
y_valid = valid_df["retained_d7"].astype(int).copy()
y_test = test_df["retained_d7"].astype(int).copy()

# 범주형 컬럼 타입 통일
for col in categorical_features:
    X_train[col] = X_train[col].astype("string")
    X_valid[col] = X_valid[col].astype("string")
    X_test[col] = X_test[col].astype("string")

print("수치형 피처:", len(numeric_features))
print("범주형 피처:", len(categorical_features))

print("\nX_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("X_test:", X_test.shape)

### 4-2. 전처리 Pipeline

수치형은 중앙값 대치 + 결측 indicator + 표준화, 범주형은 최빈값 대치 + One-Hot Encoding을 적용한다. Pipeline 내부에서 학습되므로 Validation/Test 정보가 전처리에 유입되지 않는다.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
            ),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            ),
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features,
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features,
        ),
    ],
    remainder="drop",
)

print("전처리 Pipeline 생성 완료")

### 4-3. 공통 평가 함수


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)


def evaluate_binary_classifier(
    model,
    X,
    y,
    model_name,
    threshold=0.5,
):
    positive_probability = model.predict_proba(X)[:, 1]

    prediction = (
        positive_probability >= threshold
    ).astype(int)

    return {
        "model": model_name,
        "roc_auc": roc_auc_score(
            y,
            positive_probability
        ),
        "pr_auc": average_precision_score(
            y,
            positive_probability
        ),
        "log_loss": log_loss(
            y,
            positive_probability
        ),
        "brier_score": brier_score_loss(
            y,
            positive_probability
        ),
        "accuracy": accuracy_score(
            y,
            prediction
        ),
        "precision": precision_score(
            y,
            prediction,
            zero_division=0,
        ),
        "recall": recall_score(
            y,
            prediction,
            zero_division=0,
        ),
        "f1": f1_score(
            y,
            prediction,
            zero_division=0,
        ),
    }

## 5. Baseline: Dummy Classifier


In [ ]:
from sklearn.dummy import DummyClassifier

dummy_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            DummyClassifier(
                strategy="prior"
            ),
        ),
    ]
)

dummy_pipeline.fit(
    X_train,
    y_train
)

dummy_valid_result = (
    evaluate_binary_classifier(
        model=dummy_pipeline,
        X=X_valid,
        y=y_valid,
        model_name="Dummy",
    )
)

dummy_valid_result_df = pd.DataFrame(
    [dummy_valid_result]
).round(4)

dummy_valid_result_df

## 6. Logistic Regression


In [ ]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression

logistic_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(preprocessor)
        ),
        (
            "model",
            LogisticRegression(
                penalty="l2",
                C=1.0,
                solver="liblinear",
                max_iter=2000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

logistic_pipeline.fit(
    X_train,
    y_train
)

print("Logistic Regression 학습 완료")


### 6-1. Train / Validation 성능


In [ ]:
logistic_train_result = (
    evaluate_binary_classifier(
        model=logistic_pipeline,
        X=X_train,
        y=y_train,
        model_name="Logistic - Train",
    )
)

logistic_valid_result = (
    evaluate_binary_classifier(
        model=logistic_pipeline,
        X=X_valid,
        y=y_valid,
        model_name="Logistic - Validation",
    )
)

logistic_performance = pd.DataFrame([
    logistic_train_result,
    logistic_valid_result,
]).round(4)

logistic_performance

### 6-2. Dummy 대비 개선 확인


In [ ]:
valid_model_comparison = pd.DataFrame([
    dummy_valid_result,
    logistic_valid_result,
]).round(4)

valid_model_comparison

### 6-3. Validation 예측확률 분포


In [ ]:
logistic_valid_probability = (
    logistic_pipeline
    .predict_proba(X_valid)[:, 1]
)

pd.Series(
    logistic_valid_probability,
    name="retention_probability"
).describe(
    percentiles=[0.01, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99]
).round(4)

## 7. Random Forest


In [ ]:
from sklearn.ensemble import RandomForestClassifier

random_forest_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(preprocessor)
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=500,
                max_depth=8,
                min_samples_leaf=20,
                max_features="sqrt",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

random_forest_pipeline.fit(
    X_train,
    y_train
)

print("Random Forest 학습 완료")


In [ ]:
rf_train_result = evaluate_binary_classifier(
    model=random_forest_pipeline,
    X=X_train,
    y=y_train,
    model_name="Random Forest - Train",
)

rf_valid_result = evaluate_binary_classifier(
    model=random_forest_pipeline,
    X=X_valid,
    y=y_valid,
    model_name="Random Forest - Validation",
)

pd.DataFrame([
    rf_train_result,
    rf_valid_result,
]).round(4)

## 8. LightGBM


In [ ]:
!pip install -q lightgbm

In [ ]:
from lightgbm import LGBMClassifier

lightgbm_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(preprocessor)
        ),
        (
            "model",
            LGBMClassifier(
                objective="binary",
                n_estimators=400,
                learning_rate=0.03,
                num_leaves=15,
                min_child_samples=30,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.1,
                reg_lambda=1.0,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                verbosity=-1,
            ),
        ),
    ]
)

lightgbm_pipeline.fit(
    X_train,
    y_train
)

print("LightGBM 학습 완료")


In [ ]:
lgbm_train_result = evaluate_binary_classifier(
    model=lightgbm_pipeline,
    X=X_train,
    y=y_train,
    model_name="LightGBM - Train",
)

lgbm_valid_result = evaluate_binary_classifier(
    model=lightgbm_pipeline,
    X=X_valid,
    y=y_valid,
    model_name="LightGBM - Validation",
)

pd.DataFrame([
    lgbm_train_result,
    lgbm_valid_result,
]).round(4)

## 9. 후보 모델 Validation 성능 비교

모델 선택은 **Validation 성능**을 기준으로 진행하고, Test는 아직 보지 않는다. 확률 기반 위험점수 산출이 목적이므로 Log Loss와 Brier Score를 특히 함께 확인한다.


In [ ]:
valid_results = pd.DataFrame([
    dummy_valid_result,
    logistic_valid_result,
    rf_valid_result,
    lgbm_valid_result,
]).round(4)

valid_results

In [ ]:
valid_results[
    [
        "model",
        "roc_auc",
        "pr_auc",
        "log_loss",
        "brier_score",
    ]
].sort_values(
    by="log_loss"
)

### 9-1. 과적합 정도 비교


In [ ]:
overfit_comparison = pd.DataFrame({
    "model": [
        "Logistic",
        "Random Forest",
        "LightGBM",
    ],
    "train_roc_auc": [
        logistic_train_result["roc_auc"],
        rf_train_result["roc_auc"],
        lgbm_train_result["roc_auc"],
    ],
    "valid_roc_auc": [
        logistic_valid_result["roc_auc"],
        rf_valid_result["roc_auc"],
        lgbm_valid_result["roc_auc"],
    ],
})

overfit_comparison["roc_auc_gap"] = (
    overfit_comparison["train_roc_auc"]
    - overfit_comparison["valid_roc_auc"]
)

overfit_comparison.round(4)

**현재 결과 해석**
- Random Forest가 Validation에서 후보 모델 중 가장 좋은 ROC-AUC / Log Loss를 보였다.
- LightGBM은 Train 성능이 매우 높지만 Validation 성능이 떨어져 과적합 신호가 가장 크다.
- 따라서 이후 Ablation과 튜닝은 Random Forest를 중심으로 진행한다.


## 10. Ablation Test

피처 그룹을 하나씩 제거해 성능 변화를 확인한다. 단순히 중요도만 보는 것보다, 특정 피처 그룹이 실제 일반화 성능에 기여하는지 확인하는 목적이다.


In [ ]:
def build_preprocessor(selected_features):
    selected_categorical = [
        col
        for col in categorical_features
        if col in selected_features
    ]

    selected_numeric = [
        col
        for col in numeric_features
        if col in selected_features
    ]

    transformers = []

    if selected_numeric:
        numeric_transformer = Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median",
                        add_indicator=True,
                    ),
                ),
                (
                    "scaler",
                    StandardScaler(),
                ),
            ]
        )

        transformers.append(
            (
                "numeric",
                numeric_transformer,
                selected_numeric,
            )
        )

    if selected_categorical:
        categorical_transformer = Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(
                        strategy="most_frequent"
                    ),
                ),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore"
                    ),
                ),
            ]
        )

        transformers.append(
            (
                "categorical",
                categorical_transformer,
                selected_categorical,
            )
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )

### 10-1. Random Forest 생성 함수


In [ ]:
def build_random_forest_pipeline(selected_features):
    return Pipeline(
        steps=[
            (
                "preprocessor",
                build_preprocessor(
                    selected_features
                ),
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=500,
                    max_depth=8,
                    min_samples_leaf=20,
                    max_features="sqrt",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    )


### 10-2. Ablation 조합 정의


In [ ]:
ablation_cases = {
    "All Features": feature_columns,

    "Without Basic": [
        col for col in feature_columns
        if col not in basic_features
    ],

    "Without PING": [
        col for col in feature_columns
        if col not in ping_features
    ],

    "Without Hint": [
        col for col in feature_columns
        if col not in hint_features
    ],

    "Without Category": [
        col for col in feature_columns
        if col not in category_features
    ],
}

for case_name, selected_features in ablation_cases.items():
    print(
        case_name,
        "→",
        len(selected_features),
        "개 피처"
    )

### 10-3. Ablation 실행


In [ ]:
ablation_results = []

for case_name, selected_features in ablation_cases.items():
    print(f"학습 중: {case_name}")

    ablation_model = (
        build_random_forest_pipeline(
            selected_features
        )
    )

    ablation_model.fit(
        X_train[selected_features],
        y_train,
    )

    result = evaluate_binary_classifier(
        model=ablation_model,
        X=X_valid[selected_features],
        y=y_valid,
        model_name=case_name,
    )

    result["feature_count"] = len(
        selected_features
    )

    ablation_results.append(result)

print("Ablation Test 완료")

### 10-4. Ablation 결과 비교


In [ ]:
ablation_results_df = pd.DataFrame(
    ablation_results
)

full_result = (
    ablation_results_df
    .loc[
        ablation_results_df["model"]
        == "All Features"
    ]
    .iloc[0]
)

ablation_results_df["roc_auc_change"] = (
    ablation_results_df["roc_auc"]
    - full_result["roc_auc"]
)

ablation_results_df["pr_auc_change"] = (
    ablation_results_df["pr_auc"]
    - full_result["pr_auc"]
)

ablation_results_df["log_loss_change"] = (
    ablation_results_df["log_loss"]
    - full_result["log_loss"]
)

ablation_results_df[
    [
        "model",
        "feature_count",
        "roc_auc",
        "roc_auc_change",
        "pr_auc",
        "pr_auc_change",
        "log_loss",
        "log_loss_change",
        "brier_score",
    ]
].round(4)

**현재 Ablation 결과**
- `Without Basic`: 성능 하락 폭이 가장 커 기본 사용자 피처는 제거하지 않는 편이 낫다.
- `Without Hint`: ROC-AUC +0.0065, Log Loss -0.0023으로 소폭 개선됐다.
- `Without PING`, `Without Category`: 변화가 매우 작다.

Ablation 차이는 크지 않으므로 **'힌트 피처가 불필요하다'고 단정하지 않고**, 최종 후보 조합 선택을 위한 실험 결과로만 사용한다.


## 11. 최종 후보 피처 선택

현재 실행 결과에서는 `Without Hint`가 All Features 대비 Validation ROC-AUC와 Log Loss가 소폭 개선되어, 다음 튜닝 단계에서는 힌트 피처를 제외한 조합을 후보로 사용한다.

> 이 선택은 **힌트가 중요하지 않다는 뜻이 아니라**, 현재 표본·시점·모델에서 해당 피처 그룹을 제거했을 때 일반화 성능이 조금 더 나았다는 뜻이다.


In [ ]:
selected_features = [
    col
    for col in feature_columns
    if col not in hint_features
]

print("튜닝 대상 피처 수:", len(selected_features))
print(selected_features)

## 12. 시간순 교차검증 기반 하이퍼파라미터 튜닝

Train 구간 내부에서도 `TimeSeriesSplit`을 사용해 시간 순서를 유지한다. 최적화 기준은 `neg_log_loss`다.


In [ ]:
from sklearn.model_selection import (
    RandomizedSearchCV,
    TimeSeriesSplit,
)

time_series_cv = TimeSeriesSplit(
    n_splits=3
)

tuning_pipeline = (
    build_random_forest_pipeline(
        selected_features
    )
)

# Grid Search와 모델 내부 병렬처리가
# 겹치지 않도록 모델은 단일 코어로 설정
tuning_pipeline.set_params(
    model__n_jobs=1
)

parameter_space = {
    "model__n_estimators": [
        300,
        500,
        700,
    ],
    "model__max_depth": [
        4,
        6,
        8,
        10,
    ],
    "model__min_samples_leaf": [
        10,
        20,
        30,
        40,
    ],
    "model__max_features": [
        "sqrt",
        0.5,
        0.8,
    ],
}

random_search = RandomizedSearchCV(
    estimator=tuning_pipeline,
    param_distributions=parameter_space,
    n_iter=12,
    scoring="neg_log_loss",
    cv=time_series_cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True,
    verbose=1,
)

random_search.fit(
    X_train[selected_features],
    y_train,
)

print("튜닝 완료")


### 12-1. 최적 파라미터 확인


In [ ]:
print("Best CV Log Loss:")
print(
    round(
        -random_search.best_score_,
        4
    )
)

print("\nBest Parameters:")
random_search.best_params_

### 12-2. 튜닝 모델 성능


In [ ]:
tuned_rf_pipeline = (
    random_search.best_estimator_
)

tuned_rf_train_result = (
    evaluate_binary_classifier(
        model=tuned_rf_pipeline,
        X=X_train[selected_features],
        y=y_train,
        model_name="Tuned RF - Train",
    )
)

tuned_rf_valid_result = (
    evaluate_binary_classifier(
        model=tuned_rf_pipeline,
        X=X_valid[selected_features],
        y=y_valid,
        model_name="Tuned RF - Validation",
    )
)

pd.DataFrame([
    tuned_rf_train_result,
    tuned_rf_valid_result,
]).round(4)

### 12-3. 튜닝 전·후 비교

튜닝 결과가 항상 더 좋아지는 것은 아니다. Validation 기준으로 튜닝 전 모델과 비교해 최종 사양을 결정한다.


In [ ]:
untuned_without_hint_result = (
    ablation_results_df
    .loc[
        ablation_results_df["model"]
        == "Without Hint"
    ]
    .iloc[0]
    .to_dict()
)

tuning_comparison = pd.DataFrame([
    {
        "model": "RF Without Hint - Untuned",
        "roc_auc": untuned_without_hint_result[
            "roc_auc"
        ],
        "pr_auc": untuned_without_hint_result[
            "pr_auc"
        ],
        "log_loss": untuned_without_hint_result[
            "log_loss"
        ],
        "brier_score": untuned_without_hint_result[
            "brier_score"
        ],
    },
    {
        "model": "RF Without Hint - Tuned",
        "roc_auc": tuned_rf_valid_result[
            "roc_auc"
        ],
        "pr_auc": tuned_rf_valid_result[
            "pr_auc"
        ],
        "log_loss": tuned_rf_valid_result[
            "log_loss"
        ],
        "brier_score": tuned_rf_valid_result[
            "brier_score"
        ],
    },
])

tuning_comparison.round(4)

**모델 선택 결정**
- 튜닝 전 `Without Hint`: Log Loss **0.6596**
- 튜닝 후 `Without Hint`: Log Loss **0.6615**

튜닝 후 성능이 소폭 악화됐으므로 최종 모델은 **튜닝 전 Random Forest + Without Hint 피처 조합**을 사용한다.


## 13. 최종 모델 학습 데이터 구성

모델·피처·하이퍼파라미터 선택이 끝났으므로 Train과 Validation을 합쳐 최종 학습 데이터로 사용한다.


In [ ]:
X_train_valid = pd.concat(
    [
        X_train[selected_features],
        X_valid[selected_features],
    ],
    axis=0,
    ignore_index=True,
)

y_train_valid = pd.concat(
    [
        y_train,
        y_valid,
    ],
    axis=0,
    ignore_index=True,
)

print("최종 학습 데이터:", X_train_valid.shape)
print("최종 학습 타깃:", y_train_valid.shape)
print(
    "최종 학습 유지율:",
    round(y_train_valid.mean() * 100, 2),
)

### 13-1. 최종 Random Forest 재학습 : 튜닝 전 Random Forest

현재 실행 결과에서는 `Without Hint - Untuned`가 튜닝 모델보다 Validation Log Loss가 더 낮았으므로, **튜닝 전 Random Forest 설정**으로 최종 모델을 재학습한다.


In [ ]:
# Validation 비교 결과에 따라 튜닝 전 Without Hint RF를 최종 사양으로 선택
FINAL_MODEL_VERSION = "rf_without_hint_untuned_v1"

final_rf_pipeline = (
    build_random_forest_pipeline(
        selected_features
    )
)

final_rf_pipeline.fit(
    X_train_valid,
    y_train_valid,
)

print("최종 Random Forest 재학습 완료")


## 14. Test 최종 평가

이 단계에서 처음으로 Test 성능을 확인한다. Dummy와 함께 비교해 실제로 기준 모델을 넘어서는지 확인한다.


In [ ]:
final_test_result = (
    evaluate_binary_classifier(
        model=final_rf_pipeline,
        X=X_test[selected_features],
        y=y_test,
        model_name="Final Random Forest - Test",
    )
)

final_test_result_df = pd.DataFrame(
    [final_test_result]
).round(4)

final_test_result_df

In [ ]:
final_dummy_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            build_preprocessor(
                selected_features
            ),
        ),
        (
            "model",
            DummyClassifier(
                strategy="prior"
            ),
        ),
    ]
)

final_dummy_pipeline.fit(
    X_train_valid,
    y_train_valid,
)

dummy_test_result = (
    evaluate_binary_classifier(
        model=final_dummy_pipeline,
        X=X_test[selected_features],
        y=y_test,
        model_name="Dummy - Test",
    )
)

test_comparison = pd.DataFrame([
    dummy_test_result,
    final_test_result,
]).round(4)

test_comparison

**현재 Test 결과**
- Final Random Forest: ROC-AUC **0.6737**, PR-AUC **0.6650**, Log Loss **0.6521**
- Dummy: ROC-AUC **0.5000**, Log Loss **0.6932**

Test에서도 Dummy 대비 확률 예측 품질이 개선되어, 프로토타입 수준의 위험도 스코어링 모델로 사용할 근거는 확보됐다. (다만 운영 적용 전에는 시점별 재검증과 calibration 확인이 추가로 필요하다.)


## 프로토타입 결론

현재 프로토타입은 다음 흐름을 모두 포함한다.

- 사용자 단위 Training Mart 로드 및 QA
- 시간순 Train / Validation / Test 분할
- Train-only 전처리 Pipeline
- Dummy / Logistic / Random Forest / LightGBM 비교
- Ablation Test
- 시간순 교차검증 기반 튜닝
- 최종 Test 평가

# 운영 시뮬레이션

**굵은 텍스트**## 1. 사용자별 유지확률 / 이탈 위험점수 생성

`risk_score = 1 - retention_probability`로 정의한다. 이 값은 절대적인 이탈 확률이라기보다 현재 모델이 산출한 상대적 위험 신호로 해석한다.


In [ ]:
test_retention_probability = (
    final_rf_pipeline
    .predict_proba(
        X_test[selected_features]
    )[:, 1]
)

test_risk_score = (
    1 - test_retention_probability
)

test_prediction = (
    test_df[
        [
            "user_id",
            "first_ping_at",
            "first_ping_at_kst",
            "retained_d7",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

test_prediction = test_prediction.rename(
    columns={
        "retained_d7": "actual_retained_d7"
    }
)

test_prediction["retention_probability"] = (
    test_retention_probability
)

test_prediction["risk_score"] = (
    test_risk_score
)

test_prediction["predicted_retained_d7"] = (
    test_retention_probability >= 0.5
).astype(int)

test_prediction.head()

### 1-1. 상대적 위험구간 생성

Test 집단 내 위험점수 percentile을 사용해 HIGH(상위 20%), MEDIUM(다음 30%), LOW(하위 50%)로 구분한다.


In [ ]:
test_prediction["risk_percentile"] = (
    test_prediction["risk_score"]
    .rank(
        pct=True,
        method="average",
    )
)

test_prediction["risk_segment"] = np.select(
    [
        test_prediction["risk_percentile"] >= 0.80,
        test_prediction["risk_percentile"] >= 0.50,
    ],
    [
        "HIGH",
        "MEDIUM",
    ],
    default="LOW",
)

test_prediction["dataset_split"] = "TEST"
test_prediction["model_version"] = FINAL_MODEL_VERSION

test_prediction["scored_at"] = (
    pd.Timestamp.now(tz="UTC")
)

test_prediction.head()


### 1-2. 위험구간별 실제 D+7 결과 확인


In [ ]:
risk_segment_summary = (
    test_prediction
    .groupby(
        "risk_segment",
        as_index=False,
    )
    .agg(
        user_count=("user_id", "count"),
        avg_risk_score=("risk_score", "mean"),
        actual_retention_rate=(
            "actual_retained_d7",
            "mean",
        ),
    )
)

risk_segment_summary[
    "actual_non_retention_rate"
] = (
    1
    - risk_segment_summary[
        "actual_retention_rate"
    ]
)

risk_segment_summary[
    [
        "avg_risk_score",
        "actual_retention_rate",
        "actual_non_retention_rate",
    ]
] = (
    risk_segment_summary[
        [
            "avg_risk_score",
            "actual_retention_rate",
            "actual_non_retention_rate",
        ]
    ]
    .mul(100)
    .round(2)
)

risk_segment_summary.sort_values(
    "avg_risk_score",
    ascending=False,
)

**현재 위험구간 검증**
- HIGH: 실제 비유지율 73.97%
- MEDIUM: 실제 비유지율 55.96%
- LOW: 실제 비유지율 38.29%

위험구간이 HIGH → MEDIUM → LOW 순으로 실제 비유지율과 일관되게 정렬된다. 즉, 모델의 `risk_score`가 상대적 위험 순위를 구분하는 용도로는 의미 있게 작동한다.


### 1-3. 예측 결과 QA


In [ ]:
# 1) 기존 예측 QA
prediction_qa = pd.DataFrame({
    "check": [
        "전체 예측 사용자",
        "고유 사용자",
        "중복 사용자",
        "유지확률 결측",
        "위험점수 결측",
        "확률 범위 오류",
    ],
    "value": [
        len(test_prediction),
        test_prediction["user_id"].nunique(),
        test_prediction["user_id"].duplicated().sum(),
        test_prediction["retention_probability"].isna().sum(),
        test_prediction["risk_score"].isna().sum(),
        (~test_prediction["retention_probability"].between(0, 1)).sum(),
    ],
})

prediction_qa

### 컬럼 추가

In [ ]:
# 2) 참고용 힌트 열람 여부 추가
hint_info = training_df[
    ["user_id", "first_hint_opened_in_24h"]
].drop_duplicates(subset=["user_id"])

prediction_result = test_prediction.merge(
    hint_info,
    on="user_id",
    how="left",
    validate="one_to_one"
)

In [ ]:
# 3) BigQuery 저장용 컬럼 정리
prediction_result = prediction_result[
    [
        "user_id",
        "first_ping_at_kst",
        "actual_retained_d7",
        "retention_probability",
        "risk_score",
        "risk_segment",
        "first_hint_opened_in_24h",
        "model_version",
        "scored_at",
    ]
].copy()

prediction_result.head()

In [ ]:
final_prediction_qa = pd.DataFrame({
    "check": [
        "전체 저장 사용자",
        "고유 사용자",
        "중복 사용자",
        "유지확률 결측",
        "위험점수 결측",
        "힌트 여부 결측",
        "확률 범위 오류",
    ],
    "value": [
        len(prediction_result),
        prediction_result["user_id"].nunique(),
        prediction_result["user_id"].duplicated().sum(),
        prediction_result["retention_probability"].isna().sum(),
        prediction_result["risk_score"].isna().sum(),
        prediction_result["first_hint_opened_in_24h"].isna().sum(),
        (~prediction_result["retention_probability"].between(0, 1)).sum(),
    ],
})

final_prediction_qa

### 운영 시뮬레이션 결과

- 최종 Test cohort 727명을 대상으로 D+7 유지확률 및 이탈 위험점수를 산출하였다.
- 이탈 위험점수를 기준으로 HIGH / MEDIUM / LOW 위험구간을 생성하였다.
- 실제 D+7 비유지율은 HIGH 73.97%, MEDIUM 55.96%, LOW 38.29%로 나타나,
  모델의 위험점수가 상대적 이탈 위험 순위를 구분하는 데 의미 있게 작동함을 확인하였다.
- 최종 예측 결과는 `mart.ml_d7_prediction_result` 테이블에 저장하였다.
- `first_hint_opened_in_24h`는 모델 입력 피처가 아닌 결과 조회 및 행동 특성 확인을 위한 참고용 컬럼으로 추가하였다.

다음 단계에서는 해당 Scoring 및 BigQuery 적재 과정을 Airflow DAG로 구성하여
운영 파이프라인 자동화 과정을 구현한다.

# 빅쿼리용 테이블 생성

In [ ]:
from google.cloud import bigquery

PROJECT_ID = "your-gcp-project"
DATASET_ID = "mart"
TABLE_ID = "ml_d7_prediction_result"

table_ref = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND"
)

load_job = client.load_table_from_dataframe(
    prediction_result,
    table_ref,
    job_config=job_config,
)

load_job.result()

print(f"✅ BigQuery 적재 완료: {table_ref}")

### BigQuery 적재 결과 QA

적재 후 행 수, 고유 사용자 수, 주요 컬럼 결측 여부를 확인하고,
위험구간별 실제 D+7 유지율이 예상한 방향으로 정렬되는지 확인한다.

In [ ]:
check_query = """
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT user_id) AS user_count,
    COUNTIF(retention_probability IS NULL) AS retention_prob_null,
    COUNTIF(risk_score IS NULL) AS risk_score_null,
    COUNTIF(first_hint_opened_in_24h IS NULL) AS hint_null
FROM `your-gcp-project.mart.ml_d7_prediction_result`
"""

check_df = client.query(check_query).to_dataframe()
check_df

In [ ]:
risk_query = """
SELECT
    risk_segment,
    COUNT(*) AS user_count,
    ROUND(AVG(risk_score), 4) AS avg_risk_score,
    ROUND(AVG(actual_retained_d7), 4) AS actual_retention_rate
FROM `your-gcp-project.mart.ml_d7_prediction_result`
GROUP BY risk_segment
ORDER BY avg_risk_score DESC
"""

risk_check_df = client.query(risk_query).to_dataframe()
risk_check_df

In [ ]:
gsutil mb -p your-gcp-project -l asia-northeast3 gs://your-gcp-project-ml-artifacts

In [ ]:
import joblib
from google.cloud import storage

joblib.dump(final_rf_pipeline, "final_rf_pipeline.joblib")

storage.Client(project=PROJECT_ID).bucket("your-gcp-project-ml-artifacts") \
    .blob("d7_retention/current/final_rf_pipeline.joblib") \
    .upload_from_filename("final_rf_pipeline.joblib")

print("모델 업로드 완료")